### Heart Disease Prediction

Link del dataset: https://www.kaggle.com/datasets/johnsmith88/heart-disease-dataset/data

El dataset contiene los siguientes atributos:

- **Age**: Edad del paciente.
- **Sex**: Sexo del paciente.
- **Chest pain type**: Tipo de dolor en el pecho (4 valores posibles).
- **Resting blood pressure**: Presión arterial en reposo.
- **Serum cholestoral (mg/dl)**: Nivel de colesterol sérico en miligramos por decilitro.
- **Fasting blood sugar > 120 mg/dl**: Indica si el nivel de azúcar en sangre en ayunas es mayor a 120 mg/dl.
- **Resting electrocardiographic results**: Resultados del electrocardiograma en reposo (valores 0, 1 o 2).
- **Maximum heart rate achieved**: Frecuencia cardíaca máxima alcanzada.
- **Exercise induced angina**: Presencia de angina inducida por ejercicio.
- **Oldpeak**: Depresión del segmento ST inducida por el ejercicio en relación con el reposo.
- **Slope of the peak exercise ST segment**: Pendiente del segmento ST durante el ejercicio máximo.
- **Number of major vessels (0–3) colored by fluoroscopy**: Número de vasos sanguíneos principales coloreados por fluoroscopía.
- **Thal**: Estado de talasemia  
  - 0 = Normal  
  - 1 = Defecto fijo  
  - 2 = Defecto reversible  



Primero importamos las dependencias necesarias

In [33]:
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import torch
from torch import nn


#### Cargamos el dataset
Usando el utils.py compartido

In [34]:
class HeartData(Dataset):
    def __init__(self, file_path):
        raw_data = pd.read_csv(file_path)
        x = raw_data.values[:, :-1]
        y = raw_data.values[:, -1].astype(int)
        min_max_scaler = MinMaxScaler()
        self.x = min_max_scaler.fit_transform(x)
        self.y = y

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return torch.tensor(self.x[idx], dtype=torch.float32), torch.tensor(
            self.y[idx], dtype=torch.long
        )


def get_data(batch_s):
    dataset = HeartData("heart.csv")
    train_size = int(len(dataset) * 0.7)
    test_size = len(dataset) - train_size
    train_dataset, test_dataset = torch.utils.data.random_split(
        dataset, [train_size, test_size]
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_s, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_s, shuffle=True)
    return train_loader, test_loader


### Creamos el modelo de la red neuronal

Definimos cual es el batch_size que utilizaremos y cargamos nuestros datos del csv

In [35]:
batch_size = 50
train_loader, test_loader = get_data(batch_size)

DEfinimos el modelo de la red neuronal, en este caso será una red neuronal con 3 capas lineales, utlizando ReLU como funcion de activación

In [36]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(13, 32),
            nn.ReLU(),
            nn.Linear(32, 8),
            nn.ReLU(),
            nn.Linear(8, 2),
        )
    def forward(self, x):
        logits = self.linear_relu_stack(x)
        return logits


model = NeuralNetwork()

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

Bucle de entreno

In [37]:
def train_loop(dataloader, model, loss_fn, optimizer):
    model.train()
    for X, y in dataloader:
        pred = model(X)
        loss = loss_fn(pred, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

Bucle de test

In [38]:
def test_loop(dataloader, model):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            correct += (pred.argmax(1) == y).sum().item()
            total += y.size(0)

    print(f"Accuracy: {correct / total:.2f}")

bucle principal donde definimos el numero de epocas iterando en los batches de datos para ir actualizando los pesos del modelo

In [39]:
epochs = 20
for epoch in range(epochs):
    train_loop(train_loader, model, loss_fn, optimizer)
    print(f"Epoch {epoch + 1}/{epochs} completada")
    test_loop(test_loader, model)

Epoch 1/20 completada
Accuracy: 0.47
Epoch 2/20 completada
Accuracy: 0.52
Epoch 3/20 completada
Accuracy: 0.68
Epoch 4/20 completada
Accuracy: 0.72
Epoch 5/20 completada
Accuracy: 0.77
Epoch 6/20 completada
Accuracy: 0.81
Epoch 7/20 completada
Accuracy: 0.80
Epoch 8/20 completada
Accuracy: 0.80
Epoch 9/20 completada
Accuracy: 0.81
Epoch 10/20 completada
Accuracy: 0.81
Epoch 11/20 completada
Accuracy: 0.82
Epoch 12/20 completada
Accuracy: 0.83
Epoch 13/20 completada
Accuracy: 0.83
Epoch 14/20 completada
Accuracy: 0.84
Epoch 15/20 completada
Accuracy: 0.86
Epoch 16/20 completada
Accuracy: 0.87
Epoch 17/20 completada
Accuracy: 0.87
Epoch 18/20 completada
Accuracy: 0.87
Epoch 19/20 completada
Accuracy: 0.88
Epoch 20/20 completada
Accuracy: 0.88


y por ultimo guardamos los pesos de nuestro modelo como muestra la documentación, https://docs.pytorch.org/tutorials/beginner/basics/saveloadrun_tutorial.html,
se guarda un archivo .pth que contiene los parametros entrenables de la red neuronal

In [40]:
torch.save(model.state_dict(), "model_weights.pth")
print("Modelo guardado correctamente")

Modelo guardado correctamente
